Статистический анализатор должен состоять из следующих модулей:

Подается входная строка: “покажи все песни после 2015 года” 

1) Лемматизатор - буду использовать лемматизатор из ЛР1 -> покажи{покажи=VERB} все{весь=DET} песни{песня=NOUN} после{после=ADP} 2015{2015=ADJ} года{год=NOUN};

2) Классификатор - -> [('покажи', 'показать', 'VERB', 'VERB'), ('все', 'все', 'DET', 'QUANTIFIER'), ('песни', 'песня', 'NOUN', 'ITEM'), ('после', 'после', 'ADP', 'KW_AFTER'), ('2015', '2015', 'ADJ', 'YEAR'), ('года', 'год', 'NOUN', 'KW_YEAR')];

3) Парсер - рекурсивный спуск по грамматике, должен строить дерево/выдавать ошибку;


### Модель леммматизатора.


In [1]:
from additional_lemma_func import *

NAME_MODEL = '../lb1/lemmatizer_model2.keras'
NAME_VOCAB = '../lb1/vocabs2.json'

model, word2idx, idx2tag, idx2rule = load_model_and_vocab(NAME_VOCAB, NAME_MODEL)

2026-03-15 14:02:17.392838: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-15 14:02:17.393333: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-15 14:02:17.489903: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-15 14:02:17.703846: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-15 14:02:20.299253: W tensorflow/compiler/tf2

модель загружена
словарь: 9666 слов, 18 тегов, 893 правил


Проверим работу лемматизатора на тестовых запросах

In [2]:
tests = [
    "найди песни монеточки 2021 года",
    "выведи 5 русских исполнителей жанра хип-хоп", 
    "выведи 10 иностранных песен за 2025 год",
    "покажи все синглы нойза",
    "покажи все песни после 2015 года"
]

lemma_tests = []

for sent in tests:
    lemma_tests.append(lemmatize(sent, model, word2idx, idx2tag, idx2rule))

for sent in lemma_tests:
    print(sent)

найди{найди=SCONJ} песни{песни=NOUN} монеточки{монеточки=ADJ} 2021{2021=ADJ} года{год=NOUN}
выведи{выведи=NOUN} 5{5=NUM} русских{русский=ADJ} исполнителей{исполнитель=NOUN} жанра{жанра=PROPN} хип-хоп{хип-хоп=PROPN}
выведи{выведи=NOUN} 10{10=NUM} иностранных{иностранный=ADJ} песен{песе=NOUN} за{за=ADP} 2025{20ый=ADJ} год{год=NOUN}
покажи{покажи=VERB} все{всё=PRON} синглы{синглы=ADJ} нойза{нойза=ADJ}
покажи{покажи=VERB} все{весь=DET} песни{песня=NOUN} после{после=ADP} 2015{2015=ADJ} года{год=NOUN}


### Классификатор.

In [3]:
DSL_DICTIONARY = {
    # глаголы
    "найди": ("найти", "VERB"),
    "покажи": ("показать", "VERB"),
    "выведи": ("вывести", "VERB"),

    # квантификатор
    "все": ("все", "QUANTIFIER"),
    "всех": ("все", "QUANTIFIER"),

    # объекты - песни
    "песня": ("песня", "ITEM"),
    "песни": ("песня", "ITEM"),
    "песню": ("песня", "ITEM"),
    "песен": ("песня", "ITEM"),

    # объекты - альбомы
    "альбом": ("альбом", "ITEM"),
    "альбомы": ("альбом", "ITEM"),
    "альбомов": ("альбом", "ITEM"),

    # объекты - синглы
    "сингл": ("сингл", "ITEM"),
    "синглы": ("сингл", "ITEM"),

    # объекты - исполнители
    "исполнитель": ("исполнитель", "ITEM"),
    "исполнителя": ("исполнитель", "ITEM"),
    "исполнителей": ("исполнитель", "ITEM"),
    "исполнители": ("исполнитель", "ITEM"),

    # жанры
    "поп": ("поп", "GENRE"),
    "рок": ("рок", "GENRE"),
    "хип-хоп": ("хип-хоп", "GENRE"),
    "джаз": ("джаз", "GENRE"),
    "классика": ("классика", "GENRE"),
    "электронная": ("электронный", "GENRE"),
    "электронный": ("электронный", "GENRE"),

    # категория
    "русский": ("русский", "CATEGORY"),
    "русская": ("русский", "CATEGORY"),
    "русских": ("русский", "CATEGORY"),
    "русские": ("русский", "CATEGORY"),
    "иностранный": ("иностранный", "CATEGORY"),
    "иностранная": ("иностранный", "CATEGORY"),
    "иностранных": ("иностранный", "CATEGORY"),
    "иностранные": ("иностранный", "CATEGORY"),

    # типы альбомов
    "ep": ("EP", "ALBUM_TYPE"),
    "студийный": ("студийный", "ALBUM_TYPE"),
    "студийных": ("студийный", "ALBUM_TYPE"),
    "концертный": ("концертный", "ALBUM_TYPE"),
    "концертных": ("концертный", "ALBUM_TYPE"),

    # служебные слова
    "жанра": ("жанр", "KW_GENRE"),
    "жанр": ("жанр", "KW_GENRE"),
    "после": ("после", "KW_AFTER"),
    "до": ("до", "KW_BEFORE"),
    "между": ("между", "KW_BETWEEN"),
    "за": ("за", "KW_FOR"),
    "и": ("и", "KW_AND"),
    "года": ("год", "KW_YEAR"),
    "год": ("год", "KW_YEAR"),
    "годами": ("год", "KW_YEAR"),
    "из": ("из", "KW_FROM"),
    "альбома": ("альбом", "KW_ALBUM"),
    "название": ("название", "KW_TITLE"),
}

def classify(lemmatizer_line: str):
    tokens = re.findall(r'(\S+?)\{(\S+?)=(\S+?)\}', lemmatizer_line)

    result = []
    for word, lemma, pos in tokens:
        word_lower = word.lower()
        lemma_lower = lemma.lower()

        if word_lower in DSL_DICTIONARY:
            dsl_lemma, dsl_type = DSL_DICTIONARY[word_lower]
            result.append((word_lower, dsl_lemma, pos, dsl_type))
            continue
        
        if lemma_lower in DSL_DICTIONARY:
            dsl_lemma, dsl_type = DSL_DICTIONARY[lemma_lower]
            result.append((word_lower, dsl_lemma, pos, dsl_type))
            continue
        
        if word_lower.isdigit() and len(word_lower) == 4:
            result.append((word_lower, word_lower, pos, "YEAR"))
            continue
        
        if word_lower.isdigit():
            result.append((word_lower, word_lower, pos, "NUMBER"))
            continue

        result.append((word_lower, lemma_lower, pos, "UNKNOWN_WORD"))

    return result

classify_tests = []

for sent in lemma_tests:
    new_sent = classify(sent)
    classify_tests.append(new_sent)
    print(new_sent)

[('найди', 'найти', 'SCONJ', 'VERB'), ('песни', 'песня', 'NOUN', 'ITEM'), ('монеточки', 'монеточки', 'ADJ', 'UNKNOWN_WORD'), ('2021', '2021', 'ADJ', 'YEAR'), ('года', 'год', 'NOUN', 'KW_YEAR')]
[('выведи', 'вывести', 'NOUN', 'VERB'), ('5', '5', 'NUM', 'NUMBER'), ('русских', 'русский', 'ADJ', 'CATEGORY'), ('исполнителей', 'исполнитель', 'NOUN', 'ITEM'), ('жанра', 'жанр', 'PROPN', 'KW_GENRE'), ('хип-хоп', 'хип-хоп', 'PROPN', 'GENRE')]
[('выведи', 'вывести', 'NOUN', 'VERB'), ('10', '10', 'NUM', 'NUMBER'), ('иностранных', 'иностранный', 'ADJ', 'CATEGORY'), ('песен', 'песня', 'NOUN', 'ITEM'), ('за', 'за', 'ADP', 'KW_FOR'), ('2025', '2025', 'ADJ', 'YEAR'), ('год', 'год', 'NOUN', 'KW_YEAR')]
[('покажи', 'показать', 'VERB', 'VERB'), ('все', 'все', 'PRON', 'QUANTIFIER'), ('синглы', 'сингл', 'ADJ', 'ITEM'), ('нойза', 'нойза', 'ADJ', 'UNKNOWN_WORD')]
[('покажи', 'показать', 'VERB', 'VERB'), ('все', 'все', 'DET', 'QUANTIFIER'), ('песни', 'песня', 'NOUN', 'ITEM'), ('после', 'после', 'ADP', 'KW_AFTE

### Парсер.

In [4]:
class ParseNode:
    def __init__(self, label: str, value: str = None, children: list = None):
        self.label = label
        self.value = value
        self.children = children or []

    def view(self, indent: int = 0):
        prefix = "  " * indent
        if self.value:
            result = f"{prefix}{self.label}: \"{self.value}\"\n"
        else:
            result = f"{prefix}{self.label}\n"
        for child in self.children:
            result += child.view(indent + 1)
        return result
    

class Parser:
    def __init__(self, tokens: list[tuple]):
        self.tokens = tokens
        self.pos = 0
    
    def current(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None
    
    def current_type(self):
        token = self.current()
        return token[3] if token else None

    def current_lemma(self):
        token = self.current()
        return token[1] if token else None

    def advance(self):
        token = self.tokens[self.pos]
        self.pos += 1
        return token

    def expect(self, dsl_type: str):
        if self.current_type() == dsl_type:
            return self.advance()
        raise ParseError(self.pos, self.current(), dsl_type)

    def at_end(self):
        return self.pos >= len(self.tokens)
    
    def parse(self) -> ParseNode:
        tree = self.parse_query()
        if not self.at_end():
            raise ParseError(self.pos, self.current(), "конец запроса")
        return tree
    
    def parse_query(self):
        """QUERY::= VERB OBJECT DETAILS"""
        verb = self.parse_verb()
        obj = self.parse_object()
        details = self.parse_details()
        children = [verb, obj]
        if details:
            children.append(details)
        return ParseNode("QUERY", children=children)
    
    def parse_verb(self):
        """VERB ::= “найти” | “показать” | “вывести” """
        token = self.expect("VERB")
        return ParseNode("VERB", value=token[1])
    
    def parse_object(self):
        """OBJECT ::= QUANTIFIER MODIFIERS ITEM"""
        quant = self.parse_quantifier()
        mods = self.parse_modifiers()
        item = self.parse_item()
        children = []
        if quant:
            children.append(quant)
        children.extend(mods)
        children.append(item)
        return ParseNode("OBJECT", children=children)
    
    def parse_quantifier(self):
        """QUANTIFIER ::= “все” | NUMBER | λ"""
        if self.current_type() == "QUANTIFIER":
            token = self.advance()
            return ParseNode("QUANTIFIER", value=token[1])
        if self.current_type() == "NUMBER":
            token = self.advance()
            return ParseNode("QUANTIFIER", value=token[1])
        return None  # пусто
    
    def parse_modifiers(self):
        """MODIFIERS ::= MODIFIER MODIFIERS | λ (пре-фильтры)"""
        mods = []
        while self.current_type() in ("CATEGORY", "ALBUM_TYPE"):
            token = self.advance()
            mods.append(ParseNode("MODIFIER", value=token[1]))
        return mods
    
    def parse_item(self):
        """ITEM ::= “песня” | “альбом” | “сингл” | “исполнитель”"""
        token = self.expect("ITEM")
        return ParseNode("ITEM", value=token[1])

    def parse_details(self):
        """DETAILS ::= FILTERS | λ"""
        filters = self.parse_filters()
        if filters:
            return ParseNode("DETAILS", children=filters)
        return None
    
    def parse_filters(self):
        """FILTERS ::= FILTER | FILTER FILTERS"""
        filters = []
        while not self.at_end():
            f = self.parse_filter()
            if f:
                filters.append(f)
            else:
                break
        return filters

    def parse_filter(self):
        """FILTER ::= AUTHOR_NAME | YEAR_FILTER | GENRE_FILTER | TITLE_FILTER | ALBUM_FILTER"""
        t = self.current_type()

        # YEAR_FILTER   ::= YEAR (“год”)? |
        #              “за” YEAR (“год”)? |
        #           “после” YEAR (“год”)? |
        #               “до” YEAR (“год”)?|
        #   “между” YEAR “и” YEAR (“год”)?
        if t in ("YEAR", "KW_FOR", "KW_AFTER", "KW_BEFORE", "KW_BETWEEN"):
            return self.parse_year_filter()

        # GENRE_FILTER  ::= “жанр” GENRE (KW_GENRE)
        if t == "KW_GENRE":
            return self.parse_genre_filter()

        # TITLE_FILTER  ::= “название” TITLE_TEXT (KW_TITLE)
        if t == "KW_TITLE":
            return self.parse_title_filter()

        # ALBUM_FILTER  ::= “из” “альбом” TITLE_TEXT (KW_FROM)
        if t == "KW_FROM":
            return self.parse_album_filter()

        # AUTHOR_NAME ::= <имя исполнителя / группы> (UNKNOWN_WORD)
        if t == "UNKNOWN_WORD":
            token = self.advance()
            return ParseNode("AUTHOR_NAME", value=token[1])

        return None
    
    def parse_year_filter(self):
        """
        YEAR_FILTER   ::= YEAR (“год”)? |
                     “за” YEAR (“год”)? |
                  “после” YEAR (“год”)? |
                      “до” YEAR (“год”)?|
          “между” YEAR “и” YEAR (“год”)?
        """
        t = self.current_type()
        children = []

        if t in ("KW_FOR", "KW_AFTER", "KW_BEFORE"):
            token = self.advance()
            children.append(ParseNode("KEYWORD", value=token[1]))
            year_token = self.expect("YEAR")
            children.append(ParseNode("YEAR", value=year_token[1]))

        elif t == "KW_BETWEEN":
            token = self.advance()
            children.append(ParseNode("KEYWORD", value=token[1]))
            year1 = self.expect("YEAR")
            children.append(ParseNode("YEAR", value=year1[1]))
            self.expect("KW_AND")
            year2 = self.expect("YEAR")
            children.append(ParseNode("YEAR", value=year2[1]))

        elif t == "YEAR":
            year_token = self.advance()
            children.append(ParseNode("YEAR", value=year_token[1]))

        if self.current_type() == "KW_YEAR":
            self.advance()

        return ParseNode("YEAR_FILTER", children=children)

    def parse_genre_filter(self):
        """GENRE_FILTER  ::= “жанр” GENRE"""
        self.expect("KW_GENRE")
        genre_token = self.expect("GENRE")
        return ParseNode("GENRE_FILTER", value=genre_token[1])

    def parse_title_filter(self):
        """TITLE_FILTER  ::= “название” TITLE_TEXT"""
        self.expect("KW_TITLE")
        title_token = self.expect("UNKNOWN_WORD")
        return ParseNode("TITLE_FILTER", value=title_token[1])

    def parse_album_filter(self):
        """ALBUM_FILTER  ::= “из” “альбом” TITLE_TEXT"""
        self.expect("KW_FROM")
        self.expect("KW_ALBUM")
        title_token = self.expect("UNKNOWN_WORD")
        return ParseNode("ALBUM_FILTER", value=title_token[1])


class ParseError(Exception):
    def __init__(self, position: int, token: tuple, expected: str):
        self.position = position
        self.token = token
        self.expected = expected

    def __str__(self):
        if self.token:
            word, lemma, pos, dsl_type = self.token
            return (f"ошибка на позиции {self.position}: "
                    f"встречено \"{word}\" ({dsl_type}), "
                    f"ожидалось: {self.expected}")
        return (f"ошибка на позиции {self.position}: "
                f"неожиданный конец запроса, "
                f"ожидалось: {self.expected}")
    

In [5]:
for sent in classify_tests:
    print(f"запрос: {sent}")

    try:
        parser = Parser(sent)
        tree = parser.parse()
        print(f"разобрано:")
        print(tree.view())
    except ParseError as e:
        print(f"{e}")
    print()

запрос: [('найди', 'найти', 'SCONJ', 'VERB'), ('песни', 'песня', 'NOUN', 'ITEM'), ('монеточки', 'монеточки', 'ADJ', 'UNKNOWN_WORD'), ('2021', '2021', 'ADJ', 'YEAR'), ('года', 'год', 'NOUN', 'KW_YEAR')]
разобрано:
QUERY
  VERB: "найти"
  OBJECT
    ITEM: "песня"
  DETAILS
    AUTHOR_NAME: "монеточки"
    YEAR_FILTER
      YEAR: "2021"


запрос: [('выведи', 'вывести', 'NOUN', 'VERB'), ('5', '5', 'NUM', 'NUMBER'), ('русских', 'русский', 'ADJ', 'CATEGORY'), ('исполнителей', 'исполнитель', 'NOUN', 'ITEM'), ('жанра', 'жанр', 'PROPN', 'KW_GENRE'), ('хип-хоп', 'хип-хоп', 'PROPN', 'GENRE')]
разобрано:
QUERY
  VERB: "вывести"
  OBJECT
    QUANTIFIER: "5"
    MODIFIER: "русский"
    ITEM: "исполнитель"
  DETAILS
    GENRE_FILTER: "хип-хоп"


запрос: [('выведи', 'вывести', 'NOUN', 'VERB'), ('10', '10', 'NUM', 'NUMBER'), ('иностранных', 'иностранный', 'ADJ', 'CATEGORY'), ('песен', 'песня', 'NOUN', 'ITEM'), ('за', 'за', 'ADP', 'KW_FOR'), ('2025', '2025', 'ADJ', 'YEAR'), ('год', 'год', 'NOUN', 'KW_YE

### Тестирование лемматизатора, классификатора и парсера на файле запросов.


Теперь проверим работу на заготовленном файле запросов.

In [6]:
def read_queries(path: str) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

queries = read_queries("test.txt")
print(f"запросов: {len(queries)}")

for q in queries[:5]:
    print(q)

запросов: 81
найди песни монеточки 2021 года
найди песни монеточки
найди песни монеточки после 2018 года
найди песни монеточки до 2016 года
найди песни монеточки за 2020 год


In [7]:
for sent in queries:
    print(f"запрос: {sent}")

    lemma_sent = lemmatize(sent, model, word2idx, idx2tag, idx2rule)
    classify_sent = classify(lemma_sent)
    try:
        parser = Parser(classify_sent)
        tree = parser.parse()
        print(f"состояние: успех")
        print(f"разобрано:")
        print(tree.view())
    except ParseError as e:
        print(f"состояние: неудача")
        print(f"{e}")
    print()

запрос: найди песни монеточки 2021 года
состояние: успех
разобрано:
QUERY
  VERB: "найти"
  OBJECT
    ITEM: "песня"
  DETAILS
    AUTHOR_NAME: "монеточки"
    YEAR_FILTER
      YEAR: "2021"


запрос: найди песни монеточки
состояние: успех
разобрано:
QUERY
  VERB: "найти"
  OBJECT
    ITEM: "песня"
  DETAILS
    AUTHOR_NAME: "монеточки"


запрос: найди песни монеточки после 2018 года
состояние: успех
разобрано:
QUERY
  VERB: "найти"
  OBJECT
    ITEM: "песня"
  DETAILS
    AUTHOR_NAME: "монеточк"
    YEAR_FILTER
      KEYWORD: "после"
      YEAR: "2018"


запрос: найди песни монеточки до 2016 года
состояние: успех
разобрано:
QUERY
  VERB: "найти"
  OBJECT
    ITEM: "песня"
  DETAILS
    AUTHOR_NAME: "монеточк"
    YEAR_FILTER
      KEYWORD: "до"
      YEAR: "2016"


запрос: найди песни монеточки за 2020 год
состояние: успех
разобрано:
QUERY
  VERB: "найти"
  OBJECT
    ITEM: "песня"
  DETAILS
    AUTHOR_NAME: "монеточк"
    YEAR_FILTER
      KEYWORD: "за"
      YEAR: "2020"


запрос: н